In [291]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [177]:
DATASET_PATH = '../Data/Raw/raw_dataset.csv'
TEST_DATASET_PATH = '../Data/Processed/test_dataset.csv'

def load_dataset(path: str):
    if not os.path.exists(path): return None
    return pd.read_csv(path)
    
dataset = load_dataset(DATASET_PATH)
# dataset.drop(columns='Student_ID', inplace=True)
dataset

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,STU1996,23,Male,2,7.5,20.3,1.89,7.3,10.7,High,8.2,6.3,80.9,No,Average,No,Medium
1996,STU1997,21,Male,2,6.4,16.8,2.14,4.6,5.8,Moderate,4.6,3.5,76.9,Yes,Poor,No,High
1997,STU1998,19,Female,1,4.5,13.1,2.54,7.5,10.0,Low,6.7,8.1,87.9,Yes,NaN,No,High
1998,STU1999,21,Male,3,8.1,22.3,2.62,4.5,6.6,Moderate,6.5,8.4,100.0,No,Good,No,Low


In [178]:
def fix_non_real_screen_times(dataset: pd.DataFrame, drop=False):
    mask = dataset['Social_Media_Hours'] > dataset['Screen_Time_Hours']

    if drop:
        indices = dataset[mask].index
        dataset = dataset.drop(indices)
    else:
        tmp = dataset['Screen_Time_Hours'].iloc[mask]
        dataset.loc[mask, 'Screen_Time_Hours'] = dataset['Social_Media_Hours'].iloc[mask]
        dataset.loc[mask, 'Social_Media_Hours'] = tmp

    mask = dataset['Social_Media_Hours'] > dataset['Screen_Time_Hours']
    print(f'Non Logic Screen Time Counts : {mask.sum()}')


fix_non_real_screen_times(dataset)

Non Logic Screen Time Counts : 0


In [179]:
def drop_non_impacting_columns(dataset: pd.DataFrame, columns=[]):
    return dataset.drop(columns=columns)

dataset = drop_non_impacting_columns(dataset, ['Age', 'Gender', 'Student_ID'])

In [180]:
mask = ((~dataset.isna()) & (~dataset.isnull())).all(axis=1)
print(f'{((~mask).mean() * 100):.2f}% of the data contain null values')

19.70% of the data contain null values


In [181]:
dataset

,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,2,7.5,20.3,1.89,7.3,10.7,High,8.2,6.3,80.9,No,Average,No,Medium
1996,2,6.4,16.8,2.14,4.6,5.8,Moderate,4.6,3.5,76.9,Yes,Poor,No,High
1997,1,4.5,13.1,2.54,7.5,10.0,Low,6.7,8.1,87.9,Yes,NaN,No,High
1998,3,8.1,22.3,2.62,4.5,6.6,Moderate,6.5,8.4,100.0,No,Good,No,Low


In [252]:
X, y = dataset.iloc[:, :-1], dataset.iloc[:, -1:]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=50, stratify=y)

In [243]:
# def get_ages_studied_years_dist(dataset: pd.DataFrame, norm_age = 23):

#     min_age = dataset['Age'].min()
#     years = norm_age - min_age
#     fig, axes = plt.subplots(years, figsize=(10, 12))

#     for id, year in enumerate(range(min_age, norm_age)):

#         sns.barplot(data=dataset['Year_of_Study'][dataset['Age'] == year].value_counts(), ax=axes[id])
#         axes[id].set_title(f'{year} Studied Years Dist')
        
#     plt.tight_layout()

# get_ages_studied_years_dist(train_dataset)

In [244]:
def process_non_binary_gender(dataset: pd.DataFrame, drop=True):
    if drop:
        indices = dataset[dataset['Gender'] == 'Non-binary'].index
        dataset = dataset.drop(indices)
    print(f'{(dataset['Gender'] == 'Non-binary').sum()} of non binary gender exists in data')

# process_non_binary_gender(train_dataset)

In [245]:
X_train.head()

,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History
1232,3,4.7,22.6,1.76,7.0,8.2,High,NaN,7.6,87.3,No,Good,Yes
449,2,6.4,15.5,2.54,6.7,10.6,Low,NaN,7.1,87.8,Yes,Poor,No
1055,3,5.4,17.1,2.36,7.5,9.8,High,6.7,9.4,89.1,No,Good,No
3,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes
1003,2,7.8,21.5,2.64,6.6,8.7,High,7.0,8.5,79.1,No,Average,Yes


In [ ]:
encoded_cols = X_train.select_dtypes(include='str').columns

norm_cols = np.array(['Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours'])
stand_cols = np.array(['Year_of_Study', 'Financial_Stress', 'Family_Support', 'Attendance_Rate'])

encoder = ColumnTransformer(transformers=[
    ('Str_Enc', OrdinalEncoder(
        categories=[
            ['Low', 'Moderate', 'High'],
            ['No', 'Yes'],
            ['Poor', 'Average', 'Good'],
            ['No', 'Yes']
        ],
        handle_unknown='use_encoded_value',
        unknown_value=np.nan
    ), encoded_cols)
], remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

imputer = IterativeImputer().set_output(transform='pandas')

stand_norm = ColumnTransformer(transformers=[
    ('Norm', MinMaxScaler(), norm_cols),
    ('Stand', StandardScaler(), stand_cols)
], remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

preprocessor = Pipeline(steps=[
    ('Str_Enc', encoder),
    ('Imputation', imputer),
    ('Stand_Norm', stand_norm)
])

X_train = preprocessor.fit_transform(X_train).astype(np.float16)
X_test = preprocessor.transform(X_test).astype(np.float16)

In [294]:
models = {
    'LogisticRegression': {
        'model': LogisticRegression(),
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier(),
    },
    'KNeighborsClassifier': {
        'model': KNeighborsClassifier(),
    }
}

model = LogisticRegression()
model.score

for k, v in models.items():
    print(f'\t\t=== Train {k} Model ===\n')
    model = v['model']
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    print(f'\t\t=== {k} Score : {score} ===\n')
    v['score'] = score


		=== Train LogisticRegression Model ===

		=== LogisticRegression Score : 0.755 ===

		=== Train RandomForestClassifier Model ===



/home/mohamed/Documents/Github/Gho-byte/Jury-Blanc/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/mohamed/Documents/Github/Gho-byte/Jury-Blanc/venv/lib/python3.12/site-packages/sklearn/base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


		=== RandomForestClassifier Score : 0.68 ===

		=== Train KNeighborsClassifier Model ===

		=== KNeighborsClassifier Score : 0.505 ===



/home/mohamed/Documents/Github/Gho-byte/Jury-Blanc/venv/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [303]:
np.ravel(y_test).shape

(200,)

In [239]:
preprocessor.fit_transform(X_train)

,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Year_of_Study,Financial_Stress,Family_Support,Attendance_Rate,Physical_Activity,Diet_Quality,Part_Time_Job,Previous_Mental_Health_History
1232,0.208955,0.562674,0.120370,0.620253,0.452830,0.287480,-0.001670,0.584638,0.321548,2.0,2.0,0.0,1.0
449,0.462687,0.364903,0.481481,0.582278,0.679245,-0.470154,0.002619,0.310001,0.391731,0.0,0.0,1.0,0.0
1055,0.313433,0.409471,0.398148,0.683544,0.603774,0.287480,0.400300,1.573329,0.574208,2.0,2.0,0.0,0.0
3,0.179104,0.345404,0.337963,0.430380,0.396226,-1.227788,-1.233771,0.804347,-1.559368,2.0,1.0,0.0,1.0
1003,0.671642,0.532033,0.527778,0.569620,0.500000,-0.470154,0.563707,1.078983,-0.829460,2.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
719,0.582090,0.470752,0.310185,0.468354,0.462264,-0.470154,-0.743550,0.529710,-1.054047,2.0,1.0,0.0,1.0
1314,0.537313,0.278552,0.328704,0.632911,0.415094,-1.227788,1.598619,1.243765,-0.801387,1.0,1.0,0.0,0.0
1407,0.641791,0.520891,0.337963,0.430380,0.405660,0.287480,0.999459,-0.074490,-3.257807,0.0,2.0,0.0,0.0
627,0.626866,0.607242,0.606481,0.468354,0.584906,1.045114,-0.525674,1.133911,0.883015,0.0,0.0,1.0,0.0


In [241]:
preprocessor.transform(X_test).dtypes

Sleep_Hours                       float64
Study_Hours_Per_Week              float64
GPA                               float64
Social_Media_Hours                float64
Screen_Time_Hours                 float64
Year_of_Study                     float64
Financial_Stress                  float64
Family_Support                    float64
Attendance_Rate                   float64
Physical_Activity                 float64
Diet_Quality                      float64
Part_Time_Job                     float64
Previous_Mental_Health_History    float64
dtype: object

In [15]:
imputer = IterativeImputer()
after_imputation = imputer.fit_transform(train_dataset.select_dtypes(include='number'))

In [16]:
train_dataset.loc[train_dataset['Social_Media_Hours'] ]

KeyError: '[5.8, 6.6, 4.9, 5.0, nan, 6.5, 7.4, 5.9, 4.2, 7.1, 6.9, 6.7, 5.7, 3.8, 5.6, 5.1, 5.5, 4.8, 4.6, 3.7, 8.1, 2.9, 6.4, 7.2, 6.3, 7.7, 4.3, 5.2, 3.3, 6.1, 6.8, 2.4, 5.4, 5.3, 6.2, 8.2, 7.9, 4.4, 7.8, 7.5, 4.5, 4.7, 7.6, 8.4, 7.3, 3.9, 3.1, 3.5, 4.1, 3.2, 8.5, 8.8, 9.4, 3.6, 8.9, 8.3, 2.8, 8.6, 3.4, 8.7, 2.1, 9.5, 9.1, 9.2, 9.3] not in index'

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64

In [ ]:
def get_high_risk_val():
    new_dataset = dataset[dataset['Mental_Health_Risk'] == 'High']
    display(new_dataset.select_dtypes(include='number').corr())

get_high_risk_val()

,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate
Age,1.000000,0.018435,0.048553,-0.007056,-0.015724,-0.062452,-0.039581,-0.020739,-0.003162,-0.030232
Year_of_Study,0.018435,1.000000,-0.050801,0.718318,0.321004,-0.329598,-0.217967,0.019647,-0.049699,0.019653
Sleep_Hours,0.048553,-0.050801,1.000000,-0.104782,0.112759,0.058802,0.125829,0.223069,-0.184800,0.020550
Study_Hours_Per_Week,-0.007056,0.718318,-0.104782,1.000000,0.417573,-0.474891,-0.317559,0.025467,0.004948,0.113851
GPA,-0.015724,0.321004,0.112759,0.417573,1.000000,-0.167128,-0.132051,0.128553,-0.104787,0.252767
Social_Media_Hours,-0.062452,-0.329598,0.058802,-0.474891,-0.167128,1.000000,0.684803,-0.103358,0.043627,-0.096963
Screen_Time_Hours,-0.039581,-0.217967,0.125829,-0.317559,-0.132051,0.684803,1.000000,-0.073658,0.023024,-0.059972
Financial_Stress,-0.020739,0.019647,0.223069,0.025467,0.128553,-0.103358,-0.073658,1.000000,0.156577,0.054031
Family_Support,-0.003162,-0.049699,-0.184800,0.004948,-0.104787,0.043627,0.023024,0.156577,1.000000,-0.103134
Attendance_Rate,-0.030232,0.019653,0.020550,0.113851,0.252767,-0.096963,-0.059972,0.054031,-0.103134,1.000000


In [ ]:
for column in dataset.columns:
    if dataset[column].dtype == 'float64' or dataset[column].dtype == 'int64':
        new_val = dataset[column].mean()
    else:  new_val = dataset[column].mode().iloc[0]
    dataset[column] = dataset[column].fillna(new_val)

In [ ]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             2000 non-null   int64  
 1   Gender                          2000 non-null   str    
 2   Year_of_Study                   2000 non-null   int64  
 3   Sleep_Hours                     2000 non-null   float64
 4   Study_Hours_Per_Week            2000 non-null   float64
 5   GPA                             2000 non-null   float64
 6   Social_Media_Hours              2000 non-null   float64
 7   Screen_Time_Hours               2000 non-null   float64
 8   Physical_Activity               2000 non-null   str    
 9   Financial_Stress                2000 non-null   float64
 10  Family_Support                  2000 non-null   float64
 11  Attendance_Rate                 2000 non-null   float64
 12  Part_Time_Job                   2000 non-null